In [ ]:
# Setup Earth Engine initialization and define Larouco wildfire AOI

import ee
import geemap

# Connection to Google Earth Engine
ee.Authenticate()
ee.Initialize()

# Geometry definition for the area of interest (AOI)
center = ee.Geometry.Point([-7.0372, 42.5475])  # (longitude, latitude)
area_of_interest = center.buffer(25000)  # Buffer of 25 km around the point

# Map centered on the AOI creation
Map = geemap.Map(center=[42.5475, -7.0372], zoom=11) # (latitude, longitude)
Map.addLayer(area_of_interest, {'color': 'red'}, 'Area of Interest')

In [ ]:
# Load and filter the Sentinel-2 image collection for the AOI and date range

# Call the Sentinel-2 image collection and filter it by the area of interest and cloud coverage
collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(area_of_interest) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))

# Date filtering for the wildfire event
pre_fire = collection.filterDate('2025-08-01', '2025-08-14').median().clip(area_of_interest)  # Pre-fire period
post_fire = collection.filterDate('2025-09-01', '2025-09-15').median().clip(area_of_interest)  # Post-fire period

In [ ]:
# NVDi and NBR calculation for pre-fire and post-fire images

# Calculate NDVI for pre-fire image
pre_fire_ndvi = pre_fire.normalizedDifference(['B8', 'B4']).rename('NDVI')

# Calculate NBR for pre-fire image
pre_fire_nbr = pre_fire.normalizedDifference(['B8', 'B12']).rename('NBR')

# Calculate NDVI for post-fire image
post_fire_ndvi = post_fire.normalizedDifference(['B8', 'B4']).rename('NDVI')

# Calculate NBR for post-fire image
post_fire_nbr = post_fire.normalizedDifference(['B8', 'B12']).rename('NBR')


In [ ]:
# Change detection during the wildfire event

dNDVI = pre_fire_ndvi.subtract(post_fire_ndvi).rename('dNDVI')

dNBR = pre_fire_nbr.subtract(post_fire_nbr).rename('dNBR')